# Notebook 11: Quantum Fourier Transform and Phase Estimation

This notebook covers the circuit decomposition of the Quantum Fourier Transform (QFT).

---

## Learning Objectives
1. Construct the QFT circuit using Hadamards and controlled-phase gates.
2. Compare custom circuit synthesis with `qiskit.circuit.library.QFT`.
3. Invert the transform using `circuit.inverse()`.


---
## Real-World Applications & Modern Use Cases

The Quantum Fourier Transform is the core computational subroutine in:
- **Quantum Phase Estimation (QPE):** Calculating molecular electronic energies and solving linear systems of equations (HHL algorithm).
- **Quantum Cryptanalysis:** Enabling polynomial-time order finding in Shor's algorithm.


---
## Section 1: Step-by-Step QFT Decomposition


In [1]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT

def build_custom_qft(n=3):
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
        for j in range(i + 1, n):
            angle = np.pi / (2 ** (j - i))
            qc.cp(angle, j, i)
        qc.barrier()
        
    # Swap registers for standard endianness
    for i in range(n // 2):
        qc.swap(i, n - i - 1)
        
    return qc

custom_qft = build_custom_qft(3)
print("Custom QFT Decomposition:")
print(custom_qft.draw(output='text'))


Custom QFT Decomposition:
     ┌───┐                   ░                ░       ░    
q_0: ┤ H ├─■────────■────────░────────────────░───────░──X─
     └───┘ │P(π/2)  │        ░ ┌───┐          ░       ░  │ 
q_1: ──────■────────┼────────░─┤ H ├─■────────░───────░──┼─
                    │P(π/4)  ░ └───┘ │P(π/2)  ░ ┌───┐ ░  │ 
q_2: ───────────────■────────░───────■────────░─┤ H ├─░──X─
                             ░                ░ └───┘ ░


---
## Section 2: Comparison with Qiskit's Library QFT


In [2]:
lib_qft = QFT(3).decompose()
print("Qiskit Library QFT:")
print(lib_qft.draw(output='text'))

# Invert the transform
inverse_qft = custom_qft.inverse()
print("\nInverse QFT Circuit:")
print(inverse_qft.draw(output='text'))


Qiskit Library QFT:
                                          ┌───┐   
q_0: ────────────────────■────────■───────┤ H ├─X─
                   ┌───┐ │        │P(π/2) └───┘ │ 
q_1: ──────■───────┤ H ├─┼────────■─────────────┼─
     ┌───┐ │P(π/2) └───┘ │P(π/4)                │ 
q_2: ┤ H ├─■─────────────■──────────────────────X─
     └───┘                                        

Inverse QFT Circuit:
         ░       ░                 ░                     ┌───┐
q_0: ─X──░───────░─────────────────░──■─────────■────────┤ H ├
      │  ░       ░           ┌───┐ ░  │         │P(-π/2) └───┘
q_1: ─┼──░───────░──■────────┤ H ├─░──┼─────────■─────────────
      │  ░ ┌───┐ ░  │P(-π/2) └───┘ ░  │P(-π/4)                
q_2: ─X──░─┤ H ├─░──■──────────────░──■───────────────────────
         ░ └───┘ ░                 ░
